# مهام اليوم - تطبيقات Streamlit التفاعلية

## Task 1: اول تطبيق Streamlit + اسئلة نظرية

### كود ملف `hello.py`:

In [1]:
%%writefile hello.py
import streamlit as st

st.title("مرحبا بك في تطبيقي الاول")
st.write("انا طالبة اتعلم تحليل البيانات وبناء التطبيقات باستخدام بايثون.")
st.write(2026)

Writing hello.py


### 2. كيف يعمل Streamlit؟ وما فائدة `@st.cache_data`؟
عندما يغير المستخدم أي أداة إدخال مثل الـ slider، يقوم Streamlit بإعادة تشغيل ملف البايثون بالكامل من السطر الأول إلى الأخير. إذا كان هناك سطر يقرأ ملف CSV كبير، فسيتم تحميل الملف من جديد مع كل حركة يقوم بها المستخدم، مما يسبب بطء شديد في التطبيق. فائدة `@st.cache_data` أنها تحفظ نتيجة قراءة الملف في الذاكرة المؤقتة، فلا يعيد قراءته من القرص إلا إذا تغير الملف نفسه، وهذا يجعل التطبيق سريعاً.

### 3. لماذا يحتاج محلل البيانات تطبيقاً وليس Notebook فقط؟
المدير أو صاحب القرار غالباً لا يعرف لغة بايثون ولا يهمه رؤية الأكواد البرمجية، بل يحتاج إلى واجهة بسيطة وواضحة تحتوي على أزرار وفلاتر تمكنه من استكشاف النتائج والرسوم البيانية بنفسه بسهولة دون أن يغير في الكود عن طريق الخطأ.

## Task 2: لوحة تحكم تفاعلية لبيانات القروض (`app.py`)

In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

st.title("لوحة تحكم بيانات القروض")
st.write("تطبيق بسيط لاستعراض وتحليل بيانات طلبات القروض.")

@st.cache_data
def load_data():
    return pd.read_csv("loan_clean.csv")

df = load_data()

st.sidebar.header("تصفية البيانات")

region_col = "Property_Area" if "Property_Area" in df.columns else df.select_dtypes(include="object").columns[0]
edu_col = "Education" if "Education" in df.columns else df.select_dtypes(include="object").columns[1]
income_col = "ApplicantIncome" if "ApplicantIncome" in df.columns else df.select_dtypes(include="number").columns[0]
loan_col = "LoanAmount" if "LoanAmount" in df.columns else df.select_dtypes(include="number").columns[1]
status_col = "Loan_Status" if "Loan_Status" in df.columns else df.columns[-1]

regions = df[region_col].dropna().unique().tolist()
selected_regions = st.sidebar.multiselect("المنطقة", regions, default=regions)

edu_options = ["الكل"] + df[edu_col].dropna().unique().tolist()
selected_edu = st.sidebar.radio("التعليم", edu_options)

min_inc = int(df[income_col].min())
max_inc = int(df[income_col].max())
max_income_filter = st.sidebar.slider("الحد الاعلى للدخل", min_inc, max_inc, max_inc)

filtered_df = df[df[region_col].isin(selected_regions)]
if selected_edu != "الكل":
    filtered_df = filtered_df[filtered_df[edu_col] == selected_edu]
filtered_df = filtered_df[filtered_df[income_col] <= max_income_filter]

if len(filtered_df) == 0:
    st.warning("لا توجد بيانات مطابقة للفلاتر المختارة. يرجى تغيير الخيارات.")
else:
    col1, col2, col3 = st.columns(3)
    
    total_apps = len(filtered_df)
    approved = filtered_df[status_col].isin(["Y", "Approved", 1, "1", "نعم"]).sum()
    approval_rate = (approved / total_apps) * 100
    median_loan = filtered_df[loan_col].median()

    col1.metric("عدد الطلبات", total_apps)
    col2.metric("نسبة الموافقة", f"{approval_rate:.1f}%")
    col3.metric("وسيط مبلغ القرض", f"{median_loan:.1f}")

    c1, c2 = st.columns(2)
    with c1:
        fig1, ax1 = plt.subplots()
        sns.countplot(data=filtered_df, x=region_col, ax=ax1)
        ax1.set_title("عدد الطلبات حسب المنطقة")
        st.pyplot(fig1)

    with c2:
        fig2, ax2 = plt.subplots()
        sns.boxplot(data=filtered_df, x=edu_col, y=loan_col, ax=ax2)
        ax2.set_title("مبلغ القرض حسب التعليم")
        st.pyplot(fig2)

    num_cols = filtered_df.select_dtypes(include="number").columns.tolist()
    selected_num = st.selectbox("اختر عمود رقمي لعرض التوزيع", num_cols)
    fig3, ax3 = plt.subplots()
    sns.histplot(filtered_df[selected_num], kde=True, ax=ax3)
    ax3.set_title(f"توزيع {selected_num}")
    st.pyplot(fig3)

    with st.expander("عرض البيانات المفلترة"):
        st.dataframe(filtered_df)

Writing app.py


## Task 3: النشر على Streamlit Cloud (`requirements.txt` + سجل الأخطاء)

In [3]:
%%writefile requirements.txt
streamlit
pandas
seaborn
matplotlib

Writing requirements.txt


### سجل الأخطاء:
إذا تم رفع التطبيق على Streamlit Cloud بدون ملف `requirements.txt`، سيتوقف التطبيق عن العمل وتظهر رسالة خطأ `ModuleNotFoundError: No module named 'seaborn'` (أو اسم أي مكتبة خارجية تم استيرادها مثل pandas أو matplotlib)، لأن السيرفر السحابي لا يثبت المكتبات الإضافية تلقائياً ما لم تكن مكتوبة في هذا الملف.

## Task 4: تطبيق ميزان الفضاء (`space_scale.py`)

In [4]:
%%writefile space_scale.py
import streamlit as st
import pandas as pd

st.title("ميزان الفضاء")
st.write("احسب وزنك على الكواكب الاخرى")

GRAVITY = {
    "Mercury": 0.38,
    "Venus": 0.90,
    "Earth": 1.00,
    "Moon": 0.17,
    "Mars": 0.38,
    "Jupiter": 2.53,
    "Saturn": 1.07,
    "Uranus": 0.89,
    "Neptune": 1.14
}

earth_weight = st.number_input("ادخل وزنك على الارض (كجم)", min_value=1.0, value=60.0)
planet = st.selectbox("اختر الكوكب", list(GRAVITY.keys()))

planet_weight = round(earth_weight * GRAVITY[planet], 2)
diff = round(planet_weight - earth_weight, 2)

st.metric(label=f"وزنك على {planet}", value=f"{planet_weight} kg", delta=f"{diff} kg")

if planet_weight < earth_weight:
    jump_factor = round(1 / GRAVITY[planet], 1)
    st.success(f"وزنك اخف من الارض! تقدر تقفز اعلى بحوالي {jump_factor} مرة.")
elif planet_weight > earth_weight:
    st.warning("وزنك اثقل من الارض هنا!")
else:
    st.info("هذا هو وزنك الطبيعي على الارض.")

weights_data = {p: round(earth_weight * g, 2) for p, g in GRAVITY.items()}
chart_df = pd.DataFrame(list(weights_data.items()), columns=["Planet", "Weight"]).set_index("Planet")
st.bar_chart(chart_df)

Writing space_scale.py
